> **2026** — local-first biology (Biopython/PDB/pandas); optional paid LLM only where noted. See `UPDATE_2026.md`.

# Chapter 7 — Omics QC & Visualization (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2007.%20LangChain%20for%20Biology/LC4LSH_Chapter_7_Omics_QC_and_Visualization.ipynb)

**Learning objectives**
- QC an expression matrix (missing values, distributions)
- Run PCA for a global view and flag batch effects
- Produce MA/volcano-style plots and heatmaps
- (Optional) Gate an LLM narrative on QC evidence

> Runtime: ~5 min (local; optional paid LLM)  
> Cost: free (no LLM needed)  
> Data: small synthetic expression matrix


## Environment setup


### Secrets (optional LLM only)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

# These notebooks run locally (Biopython/pandas); a paid LLM is OPTIONAL for narrative only.
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LS_OPENAI_API_KEY", "sk-...")
print("Optional LLM provider:", API_KEY_PROVIDER, "(analysis runs without it)")
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q biopython "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4" "langchain==1.0.0" "langchain-openai==1.0.0" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter7-omics-qc"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local-first)")


## Why QC before interpretation?

Omics matrices carry technical artifacts (batch, missingness, skew). QC and visualization come **first**; any biological story must be explicitly gated on passing QC and labelled as hypothesis, not fact.


## 1. Build a small synthetic matrix


In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
genes = [f"GENE{i}" for i in range(1, 41)]
samples = [f"S{i}_{grp}" for i in range(1, 13) for grp in (["A"] if i <= 6 else ["B"])]
samples = [f"S{i}_{'A' if i<=6 else 'B'}" for i in range(1, 13)]
X = rng.normal(0, 1, size=(len(samples), len(genes)))
X[:6, :5] += 2.0   # a real group effect in 5 genes
X[:, 10:15] += np.where(np.arange(len(samples)) % 2 == 0, 1.5, -1.5)[:, None]  # batch effect
df = pd.DataFrame(X, index=samples, columns=genes)
print(df.shape, "samples x genes"); print(df.iloc[:3, :4].round(2))


## 2. QC: missingness & distribution


In [ ]:
qc = {
    "n_samples": df.shape[0], "n_features": df.shape[1],
    "missing_frac": float(df.isna().mean().mean()),
    "mean": round(float(df.values.mean()), 3),
    "std": round(float(df.values.std()), 3),
}
print(qc)
df.iloc[:, :8].plot(kind="box", figsize=(8, 3), title="feature distributions (first 8)")
import matplotlib.pyplot as plt; plt.tight_layout(); plt.show()


## 3. PCA — global view & batch check


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Xs = StandardScaler().fit_transform(df.values)
pc = PCA(n_components=2).fit_transform(Xs)
grp = [i.split("_")[1] for i in df.index]
batch = ["even" if (int(i.split("_")[0][1:]) % 2 == 0) else "odd" for i in df.index]
pdf = pd.DataFrame(pc, columns=["PC1", "PC2"]); pdf["group"] = grp; pdf["batch"] = batch
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for g, sub in pdf.groupby("group"):
    axes[0].scatter(sub.PC1, sub.PC2, label=g)
axes[0].set_title("PCA colored by group"); axes[0].legend()
for b, sub in pdf.groupby("batch"):
    axes[1].scatter(sub.PC1, sub.PC2, label=b)
axes[1].set_title("PCA colored by batch"); axes[1].legend()
plt.tight_layout(); plt.show()
print("If batch separates as strongly as group, suspect a batch effect.")


## 4. Volcano-style plot (group A vs B)


In [ ]:
from scipy import stats
a = df[[i.endswith("_A") for i in df.index]].values
b = df[[i.endswith("_B") for i in df.index]].values
logfc = a.mean(0) - b.mean(0)
pvals = stats.ttest_ind(a, b, axis=1).pvalue
neglogp = -np.log10(np.clip(pvals, 1e-12, 1))
plt.figure(figsize=(5, 4))
plt.scatter(logfc, neglogp, c=(np.abs(logfc) > 1), cmap="coolwarm")
plt.axhline(-np.log10(0.05), ls="--", c="gray"); plt.xlabel("log2 FC (A-B)"); plt.ylabel("-log10 p")
plt.title("volcano (toy)"); plt.tight_layout(); plt.show()
print("top features:", df.columns[np.argsort(-neglogp)[:5]].tolist())


## 5. Heatmap of top varying features


In [ ]:
top = df.columns[np.argsort(df.var(0).values)[-12:]]
plt.figure(figsize=(8, 4))
plt.imshow(StandardScaler().fit_transform(df[top].values), aspect="auto", cmap="vlag" if "vlag" in plt.colormaps() else "coolwarm")
plt.colorbar(label="z"); plt.yticks(range(len(df)), df.index, fontsize=6); plt.xticks(range(len(top)), top, rotation=90, fontsize=6)
plt.title("top-variance features"); plt.tight_layout(); plt.show()


## 6. Optional: LLM narrative (gated on QC evidence)


In [ ]:
# Only call an LLM when QC passes AND you explicitly want a narrative summary.
USE_LLM = False
QC_PASS = qc["missing_frac"] == 0
if USE_LLM and QC_PASS:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = (f"Given this QC summary {qc} and PCA showing separation by both group and batch, "
              "write 3 cautious bullet points. Label each as observation or hypothesis. Do not infer mechanism.")
    print(llm.invoke(prompt).content)
else:
    print("LLM narrative OFF (USE_LLM=False or QC not passed). Plots above are the evidence; interpret cautiously.")


## Limitations & safety notes

- Toy synthetic matrix; real omics need normalization, multiple-testing correction, and batch modelling.
- A volcano/PCA pattern is an **observation**, not evidence of mechanism.
- The LLM narrative is optional and gated; never let it overrule the data.
- Local/free; optional paid LLM clearly marked.


In [ ]:
# Cleanup
import gc
for _v in ("records", "df", "model", "llm", "structure", "X"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why check batch separately from group?</summary>A batch effect can mimic biology; if batch separates as strongly as group, conclusions are unreliable.</details>

<details><summary>Why is a volcano plot not proof?</summary>It shows association (effect size vs significance), not causation or mechanism.</details>

<details><summary>Why gate the LLM on QC?</summary>So the model only summarizes already-validated evidence instead of hallucinating meaning from raw artifacts.</details>

### Tasks
- **Task A** - Add normalization (e.g., log/quantile) and compare PCA before/after.
- **Task B** - Apply Benjamini-Hochberg FDR and replot the volcano with adjusted p-values.
- **Task C** - Add a second batch variable and a batch-correction (e.g., ComBat or mean-centering per batch).
- **Task D** - Turn the LLM narrative on with a strict observation/hypothesis template and verify it abstains on low-confidence features.
